In [1]:
from pathlib import Path
import polars as pl

In [2]:
PROJECT_ROOT = Path(r"C:/Users/Rahul/OneDrive/Desktop/Learning/Projects/nyc_taxi")
PROCESSED_DIR = PROJECT_ROOT / "processed"

files = [
    PROCESSED_DIR / "yellow_taxi_2023_model_ready.parquet",
    PROCESSED_DIR / "yellow_taxi_2024_model_ready.parquet",
    PROCESSED_DIR / "yellow_taxi_2025_model_ready.parquet",
]

feature_cols = [
    "VendorID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
]

target_col = "fare_amount"

In [3]:

lf = pl.scan_parquet([str(f) for f in files]).select(
    ["tpep_pickup_datetime"] + feature_cols + [target_col]
)

train_lf = lf.filter(pl.col("tpep_pickup_datetime") < pl.datetime(2025, 1, 1))

valid_lf = lf.filter(
    (pl.col("tpep_pickup_datetime") >= pl.datetime(2025, 1, 1)) &
    (pl.col("tpep_pickup_datetime") < pl.datetime(2025, 7, 1))
)

test_lf = lf.filter(
    (pl.col("tpep_pickup_datetime") >= pl.datetime(2025, 7, 1)) &
    (pl.col("tpep_pickup_datetime") < pl.datetime(2025, 12, 1))
)

train_monthly = train_lf.with_columns([
    pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
    pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
])

valid_monthly = valid_lf.with_columns([
    pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
    pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
])

test_monthly = test_lf.with_columns([
    pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
    pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
])

print('done')

done


# create the baseline samples
### train: 20,000 per month across 24 months → about 480,000

### valid: 25,000 per month across 6 months → 150,000

### test: 30,000 per month across 5 months → 150,000

In [5]:
def monthly_balanced_sample(split_lf, per_month_n, seed=42):
    month_keys = (
        split_lf
        .select([
            pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
            pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
        ])
        .unique()
        .sort(["pickup_year", "pickup_month"])
        .collect()
    )

    sampled_parts = []

    for row in month_keys.iter_rows(named=True):
        y = row["pickup_year"]
        m = row["pickup_month"]

        month_df = (
            split_lf
            .filter(
                (pl.col("tpep_pickup_datetime").dt.year() == y) &
                (pl.col("tpep_pickup_datetime").dt.month() == m)
            )
            .collect()
        )

        n_take = min(len(month_df), per_month_n)
        month_sample = month_df.sample(n=n_take, seed=seed)

        sampled_parts.append(month_sample)

    return pl.concat(sampled_parts, how="vertical")

train_sample_pl = monthly_balanced_sample(train_lf, per_month_n=20_000, seed=42)
valid_sample_pl = monthly_balanced_sample(valid_lf, per_month_n=25_000, seed=42)
test_sample_pl  = monthly_balanced_sample(test_lf,  per_month_n=30_000, seed=42)

train_sample_pl.shape, valid_sample_pl.shape, test_sample_pl.shape

((480000, 12), (150000, 12), (150000, 12))

In [8]:
train_df = train_sample_pl.to_pandas()
valid_df = valid_sample_pl.to_pandas()
test_df  = test_sample_pl.to_pandas()

def show_month_counts(df, name):
    out = (
        pl.from_pandas(df)
        .with_columns([
            pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
            pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
        ])
        .group_by(["pickup_year", "pickup_month"])
        .agg(pl.len().alias("row_count"))
        .sort(["pickup_year", "pickup_month"])
    )
    print(f"\n=== {name} monthly counts ===")
    print(out)

show_month_counts(train_df, "train_sample")
show_month_counts(valid_df, "valid_sample")
show_month_counts(test_df, "test_sample")


=== train_sample monthly counts ===
shape: (24, 3)
┌─────────────┬──────────────┬───────────┐
│ pickup_year ┆ pickup_month ┆ row_count │
│ ---         ┆ ---          ┆ ---       │
│ i32         ┆ i8           ┆ u32       │
╞═════════════╪══════════════╪═══════════╡
│ 2023        ┆ 1            ┆ 20000     │
│ 2023        ┆ 2            ┆ 20000     │
│ 2023        ┆ 3            ┆ 20000     │
│ 2023        ┆ 4            ┆ 20000     │
│ 2023        ┆ 5            ┆ 20000     │
│ …           ┆ …            ┆ …         │
│ 2024        ┆ 8            ┆ 20000     │
│ 2024        ┆ 9            ┆ 20000     │
│ 2024        ┆ 10           ┆ 20000     │
│ 2024        ┆ 11           ┆ 20000     │
│ 2024        ┆ 12           ┆ 20000     │
└─────────────┴──────────────┴───────────┘

=== valid_sample monthly counts ===
shape: (6, 3)
┌─────────────┬──────────────┬───────────┐
│ pickup_year ┆ pickup_month ┆ row_count │
│ ---         ┆ ---          ┆ ---       │
│ i32         ┆ i8           ┆ u32   

# 5. Define features and target

In [11]:
print(train_df.columns)

Index(['tpep_pickup_datetime', 'VendorID', 'passenger_count', 'trip_distance',
       'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID',
       'pickup_hour', 'pickup_weekday', 'pickup_month', 'fare_amount'],
      dtype='object')


In [12]:
numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
]

categorical_features = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
]

target_col = "fare_amount"
print('done')

done


# Step 6: split into X and y

In [13]:
X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df[target_col].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df[target_col].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df[target_col].copy()

print('X_train.shape', X_train.shape)
print('y_train.shape', y_train.shape)
print('X_valid.shape', X_valid.shape)
print('y_valid.shape', y_valid.shape)
print('X_test.shape', X_test.shape)
print('y_test.shape', y_test.shape)

X_train.shape (480000, 10)
y_train.shape (480000,)
X_valid.shape (150000, 10)
y_valid.shape (150000,)
X_test.shape (150000, 10)
y_test.shape (150000,)


# Step 7: build the preprocessing + XGBoost pipeline

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.preprocessing import FunctionTransformer

In [18]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
print('model defined ')

model defined 


# Step 8: fit

In [19]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['trip_distance',
                                                   'passenger_count',
                                                   'pickup_hour',
                                                   'pickup_weekday',
                                                   'pickup_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='MISSING',
                                                                                 strategy='constant')),
                                                                  ('to_string',
                                                                   FunctionTransformer(func=<fun...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.08,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=-1,
                              num_parallel_tree=None, ...))])

# Step 9: evaluate on valid and test

In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

valid_pred = pipeline.predict(X_valid)
test_pred = pipeline.predict(X_test)

valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("VALID  MAE :", round(valid_mae, 4))
print("VALID RMSE :", round(valid_rmse, 4))
print("TEST   MAE :", round(test_mae, 4))
print("TEST  RMSE :", round(test_rmse, 4))

VALID  MAE : 2.2227
VALID RMSE : 4.7689
TEST   MAE : 2.835
TEST  RMSE : 5.7415


# Step 10 baseline error analysis

In [22]:
import pandas as pd
import numpy as np

valid_eval = X_valid.copy()
valid_eval["actual_fare"] = y_valid.values
valid_eval["pred_fare"] = valid_pred
valid_eval["abs_error"] = np.abs(valid_eval["actual_fare"] - valid_eval["pred_fare"])
valid_eval["split"] = "valid"

test_eval = X_test.copy()
test_eval["actual_fare"] = y_test.values
test_eval["pred_fare"] = test_pred
test_eval["abs_error"] = np.abs(test_eval["actual_fare"] - test_eval["pred_fare"])
test_eval["split"] = "test"

## Error by Fare bucket

In [23]:
fare_bins = [0, 10, 20, 40, 80, 150, np.inf]
fare_labels = ["0-10", "10-20", "20-40", "40-80", "80-150", "150+"]

for df, name in [(valid_eval, "valid"), (test_eval, "test")]:
    tmp = df.copy()
    tmp["fare_bucket"] = pd.cut(tmp["actual_fare"], bins=fare_bins, labels=fare_labels, right=False)

    summary = (
        tmp.groupby("fare_bucket", dropna=False)
        .agg(
            row_count=("actual_fare", "size"),
            mean_actual_fare=("actual_fare", "mean"),
            mae=("abs_error", "mean"),
        )
        .reset_index()
    )
    print(f"\n=== Error by fare bucket: {name} ===")
    print(summary)


=== Error by fare bucket: valid ===
  fare_bucket  row_count  mean_actual_fare        mae
0        0-10      40146          7.346012   1.365805
1       10-20      66094         14.028371   1.746832
2       20-40      29962         27.245408   3.317847
3       40-80      12603         56.932333   3.460728
4      80-150       1085         95.390000  11.907986
5        150+        110        217.222364  65.156680

=== Error by fare bucket: test ===
  fare_bucket  row_count  mean_actual_fare        mae
0        0-10      37259          7.073887   2.730394
1       10-20      63377         14.173424   2.014086
2       20-40      33662         27.440283   3.557066
3       40-80      14343         56.848459   3.756070
4      80-150       1225         96.709714  12.267116
5        150+        134        207.197910  53.940238


C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\2576772194.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("fare_bucket", dropna=False)
C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\2576772194.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("fare_bucket", dropna=False)


## Error by Trip Distance

In [27]:
dist_bins = [0, 1, 3, 5, 10, 20, np.inf]
dist_labels = ["0-1", "1-3", "3-5", "5-10", "10-20", "20+"]

for df, name in [(valid_eval, "valid"), (test_eval, "test")]:
    tmp = df.copy()
    tmp["distance_bucket"] = pd.cut(tmp["trip_distance"], bins=dist_bins, labels=dist_labels, right=False)

    summary = (
        tmp.groupby("distance_bucket", dropna=False)
        .agg(
            row_count=("actual_fare", "size"),
            mean_distance=("trip_distance", "mean"),
            mae=("abs_error", "mean"),
        )
        .reset_index()
    )
    print(f"\n=== Error by distance bucket: {name} ===")
    print(summary)


=== Error by distance bucket: valid ===
  distance_bucket  row_count  mean_distance        mae
0             0-1      31957       0.677888   1.508689
1             1-3      73288       1.766455   1.868057
2             3-5      18682       3.795967   2.827679
3            5-10      14939       7.154546   3.433210
4           10-20       9908      14.637848   3.070304
5             20+       1226     174.674201  11.210161

=== Error by distance bucket: test ===
  distance_bucket  row_count  mean_distance        mae
0             0-1      29596       0.674073   1.765087
1             1-3      69922       1.784644   2.306636
2             3-5      19874       3.812683   3.638644
3            5-10      17604       7.184275   4.448093
4           10-20      11632      14.507081   3.962159
5             20+       1372     182.479191  10.943388


C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\3855673750.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("distance_bucket", dropna=False)
C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\3855673750.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("distance_bucket", dropna=False)


## Error by Pickup Hour

In [28]:
for df, name in [(valid_eval, "valid"), (test_eval, "test")]:
    summary = (
        df.groupby("pickup_hour")
        .agg(
            row_count=("actual_fare", "size"),
            mae=("abs_error", "mean"),
        )
        .reset_index()
        .sort_values("pickup_hour")
    )
    print(f"\n=== Error by pickup hour: {name} ===")
    print(summary.head(24))


=== Error by pickup hour: valid ===
    pickup_hour  row_count       mae
0             0       4647  2.449835
1             1       3153  2.200755
2             2       1993  2.104859
3             3       1374  2.329245
4             4        972  3.153212
5             5       1114  2.400655
6             6       2213  2.092121
7             7       4342  2.190971
8             8       5835  2.438800
9             9       6303  2.258258
10           10       6590  2.293421
11           11       6906  2.159841
12           12       7608  2.284195
13           13       8108  2.346473
14           14       8684  2.300834
15           15       8927  2.440539
16           16       8922  2.345609
17           17       9767  2.246167
18           18      10288  2.158918
19           19       9083  2.049125
20           20       8834  1.836346
21           21       9210  1.894915
22           22       8472  2.181850
23           23       6655  2.296869

=== Error by pickup hour: test ===
  

## Error by origin-destination frequency

In [29]:
for df, name in [(valid_eval, "valid"), (test_eval, "test")]:
    tmp = df.copy()
    tmp["route"] = tmp["PULocationID"].astype(str) + "_" + tmp["DOLocationID"].astype(str)

    route_counts = tmp["route"].value_counts()
    tmp["route_freq"] = tmp["route"].map(route_counts)

    freq_bins = [0, 5, 20, 100, 500, np.inf]
    freq_labels = ["1-5", "6-20", "21-100", "101-500", "500+"]

    tmp["route_freq_bucket"] = pd.cut(tmp["route_freq"], bins=freq_bins, labels=freq_labels, right=True)

    summary = (
        tmp.groupby("route_freq_bucket", dropna=False)
        .agg(
            row_count=("actual_fare", "size"),
            mae=("abs_error", "mean"),
        )
        .reset_index()
    )
    print(f"\n=== Error by route frequency bucket: {name} ===")
    print(summary)


=== Error by route frequency bucket: valid ===
  route_freq_bucket  row_count       mae
0               1-5      14587  4.402260
1              6-20      16883  2.810891
2            21-100      62705  2.033475
3           101-500      52728  1.723601
4              500+       3097  1.077126

=== Error by route frequency bucket: test ===
  route_freq_bucket  row_count       mae
0               1-5      17349  5.494057
1              6-20      18162  3.734781
2            21-100      63398  2.510654
3           101-500      48671  2.052148
4              500+       2420  1.259040


C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\2772672953.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("route_freq_bucket", dropna=False)
C:\Users\Rahul\AppData\Local\Temp\ipykernel_13892\2772672953.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("route_freq_bucket", dropna=False)


### Adding PU-DO pair to improve model performance on rare routes

In [32]:
for df in [train_df, valid_df, test_df]:
    df["PU_DO_pair"] = (
        df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
    )

In [33]:
categorical_features = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "PU_DO_pair",
]

In [34]:
X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df[target_col].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df[target_col].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df[target_col].copy()

print('X_train.shape', X_train.shape)
print('y_train.shape', y_train.shape)
print('X_valid.shape', X_valid.shape)
print('y_valid.shape', y_valid.shape)
print('X_test.shape', X_test.shape)
print('y_test.shape', y_test.shape)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
print('model defined ')

X_train.shape (480000, 11)
y_train.shape (480000,)
X_valid.shape (150000, 11)
y_valid.shape (150000,)
X_test.shape (150000, 11)
y_test.shape (150000,)
model defined 


In [35]:
pipeline.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

valid_pred = pipeline.predict(X_valid)
test_pred = pipeline.predict(X_test)

valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("VALID  MAE :", round(valid_mae, 4))
print("VALID RMSE :", round(valid_rmse, 4))
print("TEST   MAE :", round(test_mae, 4))
print("TEST  RMSE :", round(test_rmse, 4))

VALID  MAE : 2.2432
VALID RMSE : 4.782
TEST   MAE : 2.8754
TEST  RMSE : 5.7831


### Removing high cardinality location data and building a simpler model with low card features

In [37]:
numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
]

categorical_features = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
]

X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df[target_col].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df[target_col].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df[target_col].copy()

print('X_train.shape', X_train.shape)
print('y_train.shape', y_train.shape)
print('X_valid.shape', X_valid.shape)
print('y_valid.shape', y_valid.shape)
print('X_test.shape', X_test.shape)
print('y_test.shape', y_test.shape)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
print('model defined ')

print('*********** Training ****************')

pipeline.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print('*********** Validation and Testing ****************')
valid_pred = pipeline.predict(X_valid)
test_pred = pipeline.predict(X_test)

valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("VALID  MAE :", round(valid_mae, 4))
print("VALID RMSE :", round(valid_rmse, 4))
print("TEST   MAE :", round(test_mae, 4))
print("TEST  RMSE :", round(test_rmse, 4))

X_train.shape (480000, 8)
y_train.shape (480000,)
X_valid.shape (150000, 8)
y_valid.shape (150000,)
X_test.shape (150000, 8)
y_test.shape (150000,)
model defined 
*********** Training ****************
*********** Validation and Testing ****************
VALID  MAE : 2.4113
VALID RMSE : 5.2296
TEST   MAE : 3.0647
TEST  RMSE : 6.2649


### Add route frequency feature

In [42]:
# create route key
for df in [train_df, valid_df, test_df]:
    df["route"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)

# route counts from TRAIN only
route_count_map = train_df["route"].value_counts().to_dict()

# map into all splits
train_df["route_train_count"] = train_df["route"].map(route_count_map).fillna(0)
valid_df["route_train_count"] = valid_df["route"].map(route_count_map).fillna(0)
test_df["route_train_count"]  = test_df["route"].map(route_count_map).fillna(0)

# optional log transform for stability
import numpy as np
train_df["route_train_count_log"] = np.log1p(train_df["route_train_count"])
valid_df["route_train_count_log"] = np.log1p(valid_df["route_train_count"])
test_df["route_train_count_log"]  = np.log1p(test_df["route_train_count"])

In [43]:
numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
    "route_train_count_log",
]

categorical_features = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
]

X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df[target_col].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df[target_col].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df[target_col].copy()

print('X_train.shape', X_train.shape)
print('y_train.shape', y_train.shape)
print('X_valid.shape', X_valid.shape)
print('y_valid.shape', y_valid.shape)
print('X_test.shape', X_test.shape)
print('y_test.shape', y_test.shape)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
print('model defined ')

print('*********** Training ****************')

pipeline.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print('*********** Validation and Testing ****************')
valid_pred = pipeline.predict(X_valid)
test_pred = pipeline.predict(X_test)

valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("VALID  MAE :", round(valid_mae, 4))
print("VALID RMSE :", round(valid_rmse, 4))
print("TEST   MAE :", round(test_mae, 4))
print("TEST  RMSE :", round(test_rmse, 4))

X_train.shape (480000, 11)
y_train.shape (480000,)
X_valid.shape (150000, 11)
y_valid.shape (150000,)
X_test.shape (150000, 11)
y_test.shape (150000,)
model defined 
*********** Training ****************
*********** Validation and Testing ****************
VALID  MAE : 2.2744
VALID RMSE : 4.9429
TEST   MAE : 3.0127
TEST  RMSE : 6.2776


# Final summary table

In [45]:
import pandas as pd

results = pd.DataFrame([
    ["baseline_pu_do",        2.2227, 4.7689, 2.8350, 5.7415],
    ["with_pu_do_pair",       2.2432, 4.7820, 2.8754, 5.7831],
    ["with_route_frequency",  2.2744, 4.9429, 3.0127, 6.2776],
    ["without_pu_do",         2.4113, 5.2296, 3.0647, 6.2649],
], columns=["experiment", "valid_mae", "valid_rmse", "test_mae", "test_rmse"])

results.sort_values("test_mae")

,experiment,valid_mae,valid_rmse,test_mae,test_rmse
0,baseline_pu_do,2.2227,4.7689,2.8350,5.7415
1,with_pu_do_pair,2.2432,4.7820,2.8754,5.7831
2,with_route_frequency,2.2744,4.9429,3.0127,6.2776
3,without_pu_do,2.4113,5.2296,3.0647,6.2649


# HYper Paramter study

In [47]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
]

categorical_features = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
]

X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df["fare_amount"].copy()

X_valid = valid_df[numeric_features + categorical_features].copy()
y_valid = valid_df["fare_amount"].copy()

X_test = test_df[numeric_features + categorical_features].copy()
y_test = test_df["fare_amount"].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("to_string", FunctionTransformer(lambda x: x.astype(str))),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

base_fixed = dict(
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)

experiments = [
    ("baseline",              {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1}),
    ("shallower_trees",       {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1}),
    ("deeper_trees",          {"n_estimators": 300, "max_depth": 10, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1}),
    ("lower_lr_more_trees",   {"n_estimators": 500, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1}),
    ("stronger_row_sampling", {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.08, "subsample": 0.6, "colsample_bytree": 0.8, "min_child_weight": 1}),
    ("stronger_col_sampling", {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.6, "min_child_weight": 1}),
    ("more_conservative",     {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5}),
]

rows = []

for name, params in experiments:
    model = XGBRegressor(**base_fixed, **params)

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)

    valid_pred = pipeline.predict(X_valid)
    test_pred = pipeline.predict(X_test)

    valid_mae = mean_absolute_error(y_valid, valid_pred)
    valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    rows.append({
        "experiment": name,
        **params,
        "valid_mae": valid_mae,
        "valid_rmse": valid_rmse,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
    })

results_hp = pd.DataFrame(rows).sort_values("test_mae")
results_hp

,experiment,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,valid_mae,valid_rmse,test_mae,test_rmse
2,deeper_trees,300,10,0.08,0.8,0.8,1,2.194209,4.757054,2.792773,5.750057
3,lower_lr_more_trees,500,8,0.05,0.8,0.8,1,2.216249,4.740184,2.829432,5.748169
0,baseline,300,8,0.08,0.8,0.8,1,2.222660,4.768881,2.834966,5.741524
5,stronger_col_sampling,300,8,0.08,0.8,0.6,1,2.221608,4.725674,2.836487,5.705676
6,more_conservative,300,8,0.08,0.8,0.8,5,2.228670,4.762762,2.845623,5.764048
4,stronger_row_sampling,300,8,0.08,0.6,0.8,1,2.233500,4.774274,2.855040,5.761494
1,shallower_trees,300,6,0.08,0.8,0.8,1,2.267126,4.782709,2.902041,5.781442
